# README
- **Author**: RUEI-HSIANG, CHANG（B12303030；AI協助）
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
重新驗證7份公開來源副本的安全轉換：第三方Google Maps API key共4個不同值／10個出現位置被替換為固定遮蔽文字，其他bytes不變。原始HTTP回應先核對後備份於使用者家目錄的私存區，不在Git repo內。這是原始來源公開表示的安全修訂，不宣稱網路來源原先不含金鑰。

## What does this file take?
固定安全對照metadata及7個原始／公開SHA。原始bytes只從私存位置讀取；沒有私存版本時只驗證已發布副本，並明記無法重做original-bytes對照。絕不下載、測試金鑰有效性、印出金鑰或撤銷第三方憑證。

## What does this file output?
逐檔驗證結果保留於Notebook，不產生新的候選人數值。已保存的7份公開副本為安全清理後來源，manifest.sha256與source_sha256均指其實際bytes；original_response_sha256保留原始身份。來源清理操作與傳播的hash維護已於本輪執行，這裡重算實際轉換來驗證。

## Known limitations
此操作不改Git歷史；commit 2a5838e仍含原始字串，GitHub警報沒有關閉。備份只在本機私存，新的clone無此原始版本。掃描只覆蓋列出的憑證模式，不是通用資安認證。研究文字不變不等於既有研究內容已全面核實。


In [1]:
from pathlib import Path
import hashlib,json,re
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').exists())
sha=lambda b:hashlib.sha256(b).hexdigest()
AUDIT_PATH='data/raw/source_capture_redaction_20261019.json'
AUDIT_SHA='491904f991121e4db009513a689f03055945ff2ccd897cefd89ffc76f4cecc83'
assert sha((ROOT/AUDIT_PATH).read_bytes())==AUDIT_SHA
audit=json.loads((ROOT/AUDIT_PATH).read_text());PATTERN=re.compile(rb'AIza[0-9A-Za-z_-]{35}')
PRIVATE=Path.home()/'.codex/private-source-backups/DSSI2026-group-3/20261019-google-maps'
checks=[]
for a in audit['files']:
 public=(ROOT/a['path']).read_bytes();assert sha(public)==a['published_redacted_sha256'] and not PATTERN.search(public)
 original=PRIVATE/a['path'];status='original_not_available_in_this_clone; only_public_version_verified'
 if original.is_file():
  raw=original.read_bytes();assert sha(raw)==a['original_sha256']
  reproduced,count=PATTERN.subn(b'[REDACTED_THIRD_PARTY_GOOGLE_API_KEY]',raw)
  assert reproduced==public and count==a['replacements']
  status='original_and_exact_key_only_transformation_verified'
 checks.append({'path':a['path'],'replacements':a['replacements'],'public_sha256':sha(public),'verification':status})
result=pd.DataFrame(checks);assert len(result)==7 and result.replacements.sum()==10
display(result);print('Published source versions verified; no key values displayed.')


,path,replacements,public_sha256,verification
0,data/raw/career_expansion_20261019/2b0861c3a2c...,1,9da1ff1992430a759190a73cb8450ddfc9ac20e1463e9a...,original_and_exact_key_only_transformation_ver...
1,data/raw/career_expansion_20261019/52d517f2cd0...,2,3385e40ba0bbe55a1ff6438be2368ec25e1b1f39d49c4b...,original_and_exact_key_only_transformation_ver...
2,data/raw/career_expansion_20261019/671b49e24de...,2,513e1b05510c3bfad66b6c81d3d0db4f1a962eb529f44d...,original_and_exact_key_only_transformation_ver...
3,data/raw/career_expansion_20261019/8b170c870f2...,2,11638f3b53a72b5bf02e1d0603df78dfc25203ca1ac991...,original_and_exact_key_only_transformation_ver...
4,data/raw/career_expansion_20261019/92d7983fa58...,1,c449936cf05a6ee8c47f66e129e0a9982db0028159c629...,original_and_exact_key_only_transformation_ver...
5,data/raw/career_expansion_20261019/f0ccfe475c8...,1,cdbb8fe941687db5571ba91d97cdeb55bcdf883a150bc5...,original_and_exact_key_only_transformation_ver...
6,data/raw/mayor_incumbency_20261019/073b4b373ff...,1,8c76981f0eb952ebea93e09e89eb0feb2b35e5130ac918...,original_and_exact_key_only_transformation_ver...


Published source versions verified; no key values displayed.
